In [1]:
%run cochain_complex.ipynb

In [3]:
g=Symp_symb(7)
C=cochain_complex(g)
gE=ext_alg(g)
Y,H,E,X,e1,e2,e3,e4,e5,e6,N=g.basis

In [233]:
def compute_ad_kernel(A,d,w=None):
    """Given a basis element A of a Lie algebra g, outputs 
    ker(ad(A)|_{C(d,w)})as a list of cochains, where C is the 
    cochain complex C(g_-,g), g=A.parent
    INPUTS:
    * 'A' - a nonnegative basis element from a Lie algebra g
    * 'd' - a degree
    * 'w' - a weight"""
    if w!=None:
        null_vecs=A.ad_mat(mod='CE',deg=d,wght=w).nullspace()
        return [C.elt({d:{w:v}}) for v in null_vecs]
    
    result={}
    for w in C.basis(d):
        temp=compute_ad_kernel(A,d,w)
        if len(temp)>0: result[w]=temp
    return result

In [234]:
def compute_eigenval(A,c,mod='CE'):
    """Computes the eigenval L so that A(c)=L*c
    INPUTS:
    * 'A' - an element of the Cartan subalgebra of a Lie algebra g
    * 'c' - an element of the Chevalley-Eilenberg complex C(g_-,g)"""

    if c==c.parent.elt({}):
        return 'Null'
    
    d=list(c.vd.keys())[0]
    w=list(c.vd[d].keys())[0]


    Ac=A.ad(c,mod)
    i=0
    temp=c.vd[d][w][i]
    while temp==0:
        i+=1
        temp=c.vd[d][w][i]
    L=Rational(Ac.vd[d][w][i],c.vd[d][w][i])
    temp=Ac-L*c
    if Ac-L*c==c.parent.elt({}): return L
    return None

In [235]:
c=C.elt_from_cd({('X','E'):1})

In [236]:
T3=C.elt_from_cd({('e_1','e_4','e_2'):-3,('e_1','e_5','e_3'):-6,('e_1','e_6','e_4'):-5,
                  ('e_2','e_3','e_2'):3,('e_2','e_4','e_3'):3,('e_2','e_5','e_4'):-1,
                  ('e_2','e_6','e_5'):-5,('e_3','e_4','e_4'):4,('e_3','e_5','e_5'):4,
                  ('e_3','e_6','e_6'):-5,('e_4','e_5','e_6'):9})
T4=C.elt_from_cd({('X','e_4','e_1'):5,('X','e_5','e_2'):8,('X','e_6','e_3'):5})
T6=C.elt_from_cd({('X','e_6','e_1'):1})

In [237]:
# X.ad(T6) should be exact,
# and X.ad(T4) should be cohomologous to something in <T3>
for a in [T3,T4,T6]:
    print('\nad(X)(',a,') :\n',X.ad(a))


ad(X)( -3*(e_1,e_4,e_2)-6*(e_1,e_5,e_3)-5*(e_1,e_6,e_4)+3*(e_2,e_3,e_2)+3*(e_2,e_4,e_3)-(e_2,e_5,e_4)-5*(e_2,e_6,e_5)+4*(e_3,e_4,e_4)+4*(e_3,e_5,e_5)-5*(e_3,e_6,e_6)+9*(e_4,e_5,e_6) ) :
 0

ad(X)( 5*(X,e_4,e_1)+8*(X,e_5,e_2)+5*(X,e_6,e_3) ) :
 -5*(X,e_3,e_1)-3*(X,e_4,e_2)+3*(X,e_5,e_3)+5*(X,e_6,e_4)

ad(X)( (X,e_6,e_1) ) :
 -(X,e_5,e_1)+(X,e_6,e_2)


In [238]:
# # Note that Y.ad(T3) is not closed
for a in [T3,T4,T6]:
    print('\nad(Y)(',a,') :\n',Y.ad(a))
print()

# However, the harmonic part is invariant in the sense that
# Harm(K) = Harm(Y.ad(K))
for a in [T3,T4,T6]:
    print(C.subspace_proj(Y.ad(a),'harmonic'))


ad(Y)( -3*(e_1,e_4,e_2)-6*(e_1,e_5,e_3)-5*(e_1,e_6,e_4)+3*(e_2,e_3,e_2)+3*(e_2,e_4,e_3)-(e_2,e_5,e_4)-5*(e_2,e_6,e_5)+4*(e_3,e_4,e_4)+4*(e_3,e_5,e_5)-5*(e_3,e_6,e_6)+9*(e_4,e_5,e_6) ) :
 -15*(e_1,e_4,e_1)-24*(e_1,e_5,e_2)-15*(e_1,e_6,e_3)+15*(e_2,e_3,e_1)+12*(e_2,e_4,e_2)-3*(e_2,e_5,e_3)-10*(e_2,e_6,e_4)+12*(e_3,e_4,e_3)+8*(e_3,e_5,e_4)-5*(e_3,e_6,e_5)+9*(e_4,e_5,e_5)

ad(Y)( 5*(X,e_4,e_1)+8*(X,e_5,e_2)+5*(X,e_6,e_3) ) :
 0

ad(Y)( (X,e_6,e_1) ) :
 0

0
0
0


In [239]:
for w in C.basis(2):
    for a in range(shape(C.subspace_basis('closed',1,w))[1]):
        c=C.elt({1:{w:(C.subspace_basis('closed',1,w).col(a))}})
        if C.cb(X.ad(c))!=C.elt({}): print('\n',c)

Note that the closed forms (and cohomology) form a $G_-$-module, but not a $G_+$

## ad(X) kernel

In [240]:
X_ker1=compute_ad_kernel(X,1)
X_ker2=compute_ad_kernel(X,2)

In [241]:
for w in X_ker1:
    print('\n',w)
    for t in X_ker1[w]:
        print(t,'-->',(compute_eigenval(H,t),compute_eigenval(E,t)))


 2
-1/2*(e_1,Y)-1/2*(e_2,H)+(e_3,X) --> (3, -1)

 1
(X,E) --> (-2, 0)
(e_1,E) --> (5, -1)
-1/2*(e_1,H)+(e_2,X) --> (5, -1)
(N,e_6) --> (5, -1)

 0
(X,X) --> (0, 0)
(e_1,X) --> (7, -1)
(e_1,e_1)+(e_2,e_2)+(e_3,e_3)+(e_4,e_4)+(e_5,e_5)+(e_6,e_6) --> (0, 0)
(N,N) --> (0, 0)

 -1
(e_1,e_2)+(e_2,e_3)+(e_3,e_4)+(e_4,e_5)+(e_5,e_6) --> (2, 0)

 -2
(e_1,e_3)+(e_2,e_4)+(e_3,e_5)+(e_4,e_6) --> (4, 0)

 -3
(e_1,e_4)+(e_2,e_5)+(e_3,e_6) --> (6, 0)

 -4
(e_1,e_5)+(e_2,e_6) --> (8, 0)

 -5
(X,e_6) --> (3, 1)
(e_1,e_6) --> (10, 0)

 -6
(X,N) --> (-2, 2)
(e_1,N) --> (5, 1)

 6
(N,X) --> (2, -2)

 7
(N,E) --> (0, -2)


In [242]:
for w in X_ker2:
    print('\n',w)
    for t in X_ker2[w]:
        print(t,'-->',(compute_eigenval(H,t),compute_eigenval(E,t)))


 3
-1/2*(X,e_1,Y)-1/2*(X,e_2,H)+(X,e_3,X) --> (1, -1)
(e_1,e_2,E) --> (8, -2)
-1/2*(e_1,e_2,H)+(e_1,e_3,X) --> (8, -2)
(e_1,N,e_5)+(e_2,N,e_6) --> (8, -2)
-1/3*(e_1,e_4,e_2)-2/3*(e_1,e_5,e_3)-5/9*(e_1,e_6,e_4)+1/3*(e_2,e_3,e_2)+1/3*(e_2,e_4,e_3)-1/9*(e_2,e_5,e_4)-5/9*(e_2,e_6,e_5)+4/9*(e_3,e_4,e_4)+4/9*(e_3,e_5,e_5)-5/9*(e_3,e_6,e_6)+(e_4,e_5,e_6) --> (1, -1)

 2
(X,e_1,E) --> (3, -1)
-1/2*(X,e_1,H)+(X,e_2,X) --> (3, -1)
(X,N,e_6) --> (3, -1)
(e_1,e_2,X) --> (10, -2)
(e_1,N,e_6) --> (10, -2)
4*(e_1,e_2,e_1)+4*(e_1,e_3,e_2)+3*(e_1,e_4,e_3)+2*(e_1,e_5,e_4)+(e_1,e_6,e_5)+(e_2,e_3,e_3)+(e_2,e_4,e_4)+(e_2,e_5,e_5)+(e_2,e_6,e_6) --> (3, -1)
5*(e_1,e_2,e_1)+5*(e_1,e_3,e_2)+3*(e_1,e_4,e_3)+(e_1,e_5,e_4)+2*(e_2,e_3,e_3)+2*(e_2,e_4,e_4)+(e_2,e_5,e_5)+(e_3,e_4,e_5)+(e_3,e_5,e_6) --> (3, -1)

 1
(X,e_1,X) --> (5, -1)
(X,e_1,e_1)+(X,e_2,e_2)+(X,e_3,e_3)+(X,e_4,e_4)+(X,e_5,e_5)+(X,e_6,e_6) --> (-2, 0)
(X,N,N) --> (-2, 0)
(e_1,e_2,e_2)+(e_1,e_3,e_3)+(e_1,e_4,e_4)+(e_1,e_5,e_5)+(e_1,e_6,e_6) --> (5, 

In [243]:
for w in X_ker1:
    print('\n',w)
    for t in X_ker1[w]:
        print(t,'-->',C.subspace_proj(t,'harmonic'))


 2
-1/2*(e_1,Y)-1/2*(e_2,H)+(e_3,X) --> 0

 1
(X,E) --> 0
(e_1,E) --> 0
-1/2*(e_1,H)+(e_2,X) --> 0
(N,e_6) --> 0

 0
(X,X) --> 0
(e_1,X) --> 0
(e_1,e_1)+(e_2,e_2)+(e_3,e_3)+(e_4,e_4)+(e_5,e_5)+(e_6,e_6) --> 0
(N,N) --> 0

 -1
(e_1,e_2)+(e_2,e_3)+(e_3,e_4)+(e_4,e_5)+(e_5,e_6) --> 0

 -2
(e_1,e_3)+(e_2,e_4)+(e_3,e_5)+(e_4,e_6) --> 0

 -3
(e_1,e_4)+(e_2,e_5)+(e_3,e_6) --> (e_1,e_4)+(e_2,e_5)+(e_3,e_6)

 -4
(e_1,e_5)+(e_2,e_6) --> 0

 -5
(X,e_6) --> 0
(e_1,e_6) --> (e_1,e_6)

 -6
(X,N) --> (X,N)
(e_1,N) --> 0

 6
(N,X) --> 0

 7
(N,E) --> 0


In [244]:
Ch1_basis=[]
j=1
Ch2_basis=[]
for w in C.basis(j):
    for i in range(shape(C.subspace_basis('harmonic',j,w))[1]):
        Ch1_basis.append(C.elt({j:{w:C.subspace_basis('harmonic',j,w).col(i)}}))

Ch2_basis=[]
j=2
for w in C.basis(j):
    for i in range(shape(C.subspace_basis('harmonic',j,w))[1]):
        Ch2_basis.append(C.elt({j:{w:C.subspace_basis('harmonic',j,w).col(i)}}))

In [245]:
for c in Ch2_basis:
    print('\n',c,'-->',(compute_eigenval(H,c),compute_eigenval(E,c)))


 -1/3*(e_1,e_4,e_2)-2/3*(e_1,e_5,e_3)-5/9*(e_1,e_6,e_4)+1/3*(e_2,e_3,e_2)+1/3*(e_2,e_4,e_3)-1/9*(e_2,e_5,e_4)-5/9*(e_2,e_6,e_5)+4/9*(e_3,e_4,e_4)+4/9*(e_3,e_5,e_5)-5/9*(e_3,e_6,e_6)+(e_4,e_5,e_6) --> (1, -1)

 3/5*(e_1,e_2,e_3)+3/5*(e_1,e_3,e_4)-2/5*(e_1,e_4,e_5)-7/5*(e_1,e_5,e_6)+(e_2,e_3,e_5)+(e_2,e_4,e_6) --> (7, -1)

 -1/2*(e_1,e_2,e_4)-1/2*(e_1,e_3,e_5)-3/2*(e_1,e_4,e_6)+(e_2,e_3,e_6) --> (9, -1)

 (e_1,e_2,e_5)+(e_1,e_3,e_6) --> (11, -1)

 (e_1,e_2,e_6) --> (13, -1)

 (X,e_4,e_1)+8/5*(X,e_5,e_2)+(X,e_6,e_3) --> (-8, 0)

 (X,e_6,e_1) --> (-12, 0)


In [246]:
Ch2_basis[0].wght_proj(3)==Ch2_basis[0]
Ch2_basis[1].wght_proj(0)==Ch2_basis[1]
Ch2_basis[2].wght_proj(-1)==Ch2_basis[2]
Ch2_basis[3].wght_proj(-2)==Ch2_basis[3]
Ch2_basis[4].wght_proj(-3)==Ch2_basis[4]
Ch2_basis[5].wght_proj(4)==Ch2_basis[5]
Ch2_basis[6].wght_proj(6)==Ch2_basis[6]

True

In [247]:
for c in Ch2_basis:
    print(Y.ad(c))

-5/3*(e_1,e_4,e_1)-8/3*(e_1,e_5,e_2)-5/3*(e_1,e_6,e_3)+5/3*(e_2,e_3,e_1)+4/3*(e_2,e_4,e_2)-1/3*(e_2,e_5,e_3)-10/9*(e_2,e_6,e_4)+4/3*(e_3,e_4,e_3)+8/9*(e_3,e_5,e_4)-5/9*(e_3,e_6,e_5)+(e_4,e_5,e_5)
24/5*(e_1,e_2,e_2)+3/5*(e_1,e_3,e_3)-43/5*(e_1,e_4,e_4)-19/5*(e_1,e_5,e_5)+7*(e_1,e_6,e_6)+5*(e_2,e_3,e_4)-2*(e_2,e_4,e_5)-(e_2,e_5,e_6)-8*(e_3,e_4,e_6)
-9/2*(e_1,e_2,e_3)-3*(e_1,e_4,e_5)+12*(e_1,e_5,e_6)+15/2*(e_2,e_3,e_5)-3/2*(e_2,e_4,e_6)
8*(e_1,e_2,e_4)-3*(e_1,e_3,e_5)-9*(e_1,e_4,e_6)-5*(e_2,e_3,e_6)
5*(e_1,e_2,e_5)-8*(e_1,e_3,e_6)
0
0


## Weight Plots

In [248]:
%matplotlib notebook

In [249]:
import matplotlib.pyplot as plt
import numpy as np

In [250]:
C2_wghts=set()
for w in C.basis(2):
    for c in C.basis(2,w):
        C2_wghts.add((compute_eigenval(H,c),compute_eigenval(E,c)))

In [251]:
x_pts=np.array([a[0] for a in C2_wghts])
y_pts=np.array([a[1] for a in C2_wghts])

In [252]:
plt.plot(x_pts,y_pts,'o')
plt.show()

<IPython.core.display.Javascript object>

In [253]:
plt.plot(0,0,'bo')
plt.grid(visible=True)
for wght in C2_wghts:
    plt.plot(wght[0],wght[1],'ro')

<IPython.core.display.Javascript object>

In [260]:
for w in C.basis(2):
    print(w,'-->',shape(C.subspace_basis('harmonic',2,w))[1])

3 --> 1
2 --> 1
1 --> 1
0 --> 1
-1 --> 2
-2 --> 2
-3 --> 1
-4 --> 1
-5 --> 1
-6 --> 0
-7 --> 0
4 --> 2
5 --> 0
6 --> 1
7 --> 0
8 --> 1
9 --> 0
10 --> 0
11 --> 0
12 --> 0
13 --> 0
14 --> 0
15 --> 0
16 --> 0
17 --> 0
18 --> 0


In [17]:
for i in range(C.subspace_basis('harmonic',2,3).shape[1]):
    print(C.elt({2:{3:C.subspace_basis('harmonic',2,3).col(i)}}),'\n')

-1/3*(e_1,e_4,e_2)-2/3*(e_1,e_5,e_3)-5/9*(e_1,e_6,e_4)+1/3*(e_2,e_3,e_2)+1/3*(e_2,e_4,e_3)-1/9*(e_2,e_5,e_4)-5/9*(e_2,e_6,e_5)+4/9*(e_3,e_4,e_4)+4/9*(e_3,e_5,e_5)-5/9*(e_3,e_6,e_6)+(e_4,e_5,e_6) 



In [27]:
for i in range(C.subspace_basis('harmonic',2,4).shape[1]):
    print(C.elt({2:{4:C.subspace_basis('harmonic',2,4).col(i)}}),'\n')

(X,e_4,e_1)+8/5*(X,e_5,e_2)+(X,e_6,e_3) 



In [26]:
for i in range(C.subspace_basis('harmonic',2,6).shape[1]):
    print(C.elt({2:{6:C.subspace_basis('harmonic',2,6).col(i)}}),'\n')

(X,e_6,e_1) 



In [4]:
f=C.basis(3,0)[6]
n=2
h=C.basis(2,4)[22]
m=2

w=gE.basis(4,10)[3]
print(w)
Gerst_prod(h,f,w)

NameError: name 'C' is not defined

In [71]:
from itertools import permutations 
for a in permutations([0,1,2]):
    print(a)

(0, 1, 2)
(0, 2, 1)
(1, 0, 2)
(1, 2, 0)
(2, 0, 1)
(2, 1, 0)


### How does ad affect weights?

In [24]:
Y_op_dict={}
for w in C.basis(2):
    for c in C.basis(2,w):
        temp1=(compute_eigenval(H,c),compute_eigenval(E,c))
        Yc=Y.ad(c)
        temp2=(compute_eigenval(H,Yc),compute_eigenval(E,Yc))
        if Yc!=C.elt({}):
            if temp1 in Y_op_dict: 
                if temp2 !=Y_op_dict[temp1]: print(temp1)
            else: Y_op_dict[temp1]=temp2
for t in Y_op_dict:
    if Y_op_dict[t]!=tuple([t[0]-2,t[1]]):print(t)

In [25]:
X_op_dict={}
for w in C.basis(2):
    for c in C.basis(2,w):
        temp1=(compute_eigenval(H,c),compute_eigenval(E,c))
        Xc=X.ad(c)
        temp2=(compute_eigenval(H,Xc),compute_eigenval(E,Xc))
        if Xc!=C.elt({}):
            if temp1 in X_op_dict: 
                if temp2 !=X_op_dict[temp1]: print(temp1)
            else: X_op_dict[temp1]=temp2

for t in X_op_dict:
    if X_op_dict[t]!=tuple([t[0]+2,t[1]]):print(t)

In [26]:
e1_op_dict={}
for w in C.basis(2):
    for c in C.basis(2,w):
        temp1=(compute_eigenval(H,c),compute_eigenval(E,c))
        e1c=e1.ad(c)
        temp2=(compute_eigenval(H,e1c),compute_eigenval(E,e1c))
        if e1c!=C.elt({}):
            if temp1 in e1_op_dict: 
                if temp2 !=e1_op_dict[temp1]: print(temp1)
            else: e1_op_dict[temp1]=temp2

for t in e1_op_dict:
    if e1_op_dict[t]!=tuple([t[0]-5,t[1]+1]):print(t)

In [27]:
for c in Ch2_basis:
    print((compute_eigenval(H,c),compute_eigenval(E,c)))

(1, -1)
(7, -1)
(9, -1)
(11, -1)
(13, -1)
(-8, 0)
(-12, 0)


In [28]:
for w in C.basis(2):
    print('\n\n',w)
    wght_set=set()
    for c in C.basis(2,w):
        wght_set.add((compute_eigenval(H,c),compute_eigenval(E,c)))
    print(wght_set)



 3
{(8, -2), (-6, 0), (1, -1)}


 2
{(10, -2), (-4, 0), (3, -1)}


 1
{(-2, 0), (5, -1)}


 0
{(7, -1), (-7, 1), (0, 0)}


 -1
{(-5, 1), (2, 0), (9, -1)}


 -2
{(4, 0), (11, -1), (-3, 1)}


 -3
{(-1, 1), (13, -1), (6, 0)}


 -4
{(1, 1), (8, 0)}


 -5
{(3, 1)}


 4
{(-8, 0), (6, -2), (-1, -1)}


 5
{(4, -2), (-10, 0), (-3, -1)}


 6
{(2, -2), (-12, 0), (-5, -1)}


 7
{(0, -2), (-7, -1), (7, -3)}


 8
{(5, -3), (-9, -1), (-2, -2)}


 9
{(-4, -2), (-11, -1), (3, -3)}


 10
{(-6, -2), (1, -3), (-13, -1)}


 11
{(-1, -3), (-8, -2)}


 12
{(-3, -3), (-10, -2)}


 13
{(-5, -3)}


 14
{(-7, -3)}


In [29]:
for w in C.basis(2):
    for i in range(shape(C.subspace_basis('harmonic',2,w))[1]):
        temp=C.elt({2:{w:C.subspace_basis('harmonic',2,w).col(i)}})
        print(temp)
        # if C.subspace_proj(X.ad(temp),'harmonic')!=C.elt({}):
        #     print(temp)

-1/3*(e_1,e_4,e_2)-2/3*(e_1,e_5,e_3)-5/9*(e_1,e_6,e_4)+1/3*(e_2,e_3,e_2)+1/3*(e_2,e_4,e_3)-1/9*(e_2,e_5,e_4)-5/9*(e_2,e_6,e_5)+4/9*(e_3,e_4,e_4)+4/9*(e_3,e_5,e_5)-5/9*(e_3,e_6,e_6)+(e_4,e_5,e_6)
3/5*(e_1,e_2,e_3)+3/5*(e_1,e_3,e_4)-2/5*(e_1,e_4,e_5)-7/5*(e_1,e_5,e_6)+(e_2,e_3,e_5)+(e_2,e_4,e_6)
-1/2*(e_1,e_2,e_4)-1/2*(e_1,e_3,e_5)-3/2*(e_1,e_4,e_6)+(e_2,e_3,e_6)
(e_1,e_2,e_5)+(e_1,e_3,e_6)
(e_1,e_2,e_6)
(X,e_4,e_1)+8/5*(X,e_5,e_2)+(X,e_6,e_3)
(X,e_6,e_1)


In [30]:
for w in X_ker2:
    print('\n',w)
    for t in X_ker2[w]:
        th=C.subspace_proj(t,'harmonic')
        if th!=C.elt({}) and th-t!=C.elt({}):
            print(t)
        # print(t,'-->',C.subspace_proj(t,'harmonic'))


 3

 2

 1

 0
2*(e_1,e_2,e_3)+2*(e_1,e_3,e_4)+(e_1,e_4,e_5)+(e_2,e_3,e_5)+(e_2,e_4,e_6)

 -1
(e_1,e_2,e_4)+(e_1,e_3,e_5)+(e_1,e_4,e_6)

 -2

 -3

 -4

 -5

 4

 5

 6

 7

 8

 9


In [31]:
for w in [0]:
    print('\n',w)
    for t in X_ker2[w]:
        # th=C.subspace_proj(t,'harmonic')
        # if th!=C.elt({}) and th-t!=C.elt({}):
        #     print(t)
        print(t,'-->',C.subspace_proj(t,'harmonic'))



 0
(X,e_1,e_2)+(X,e_2,e_3)+(X,e_3,e_4)+(X,e_4,e_5)+(X,e_5,e_6) --> 0
(e_1,e_2,e_3)+(e_1,e_3,e_4)+(e_1,e_4,e_5)+(e_1,e_5,e_6) --> 0
2*(e_1,e_2,e_3)+2*(e_1,e_3,e_4)+(e_1,e_4,e_5)+(e_2,e_3,e_5)+(e_2,e_4,e_6) --> 3/5*(e_1,e_2,e_3)+3/5*(e_1,e_3,e_4)-2/5*(e_1,e_4,e_5)-7/5*(e_1,e_5,e_6)+(e_2,e_3,e_5)+(e_2,e_4,e_6)
(e_1,e_6,N)-(e_2,e_5,N)+(e_3,e_4,N) --> 0


In [32]:
X.ad(C.subspace_proj(X_ker2[0][2],'harmonic'))

0

In [33]:
C.subspace_proj(X_ker2[0][2],'harmonic')==-Rational(7,5)*X_ker2[0][1]+X_ker2[0][2]

True

In [34]:
nW=C.elt_from_cd({('e_1','e_4','e_2'):3,('e_1','e_5','e_3'):6,('e_1','e_6','e_4'):5,('e_2','e_3','e_2'):-3,
                  ('e_2','e_4','e_3'):-3,('e_2','e_5','e_4'):1,('e_2','e_6','e_5'):5,('e_3','e_4','e_4'):-4,
                  ('e_3','e_5','e_5'):-4,('e_3','e_6','e_6'):5,('e_4','e_5','e_6'):-9})

In [35]:
alpha=g.ext_alg.elt_from_cd({('e_1','e_3'):Rational(1,3)})
alpha

1/3*(e_1,e_3)

In [36]:
for i in range(6):
    temp=alpha
    for j in range(i):
        temp=-X.ad(temp)
    print(temp,' *  e',6-i)

1/3*(e_1,e_3)  *  e 6
1/3*(e_1,e_2)  *  e 5
0  *  e 4
0  *  e 3
0  *  e 2
0  *  e 1


In [37]:
Inv_V_GL2=[C.elt_from_cd({('e_1','E'):1}),C.elt_from_cd({('e_1','X'):1}),
           C.elt_from_cd({('e_2','X'):2,('e_1','H'):-1}),C.elt_from_cd({('e_3','X'):2,('e_2','H'):-1,('e_1','Y'):-1})]

In [38]:
for A in Inv_V_GL2:
    print(C.subspace_proj(A,'harmonic'))
    print(C.subspace_proj(A.cb(),'harmonic'))

0
0
0
0
0
0
0
0


In [39]:
print(X.ad(nW,mod='CE'))
nW

0


3*(e_1,e_4,e_2)+6*(e_1,e_5,e_3)+5*(e_1,e_6,e_4)-3*(e_2,e_3,e_2)-3*(e_2,e_4,e_3)+(e_2,e_5,e_4)+5*(e_2,e_6,e_5)-4*(e_3,e_4,e_4)-4*(e_3,e_5,e_5)+5*(e_3,e_6,e_6)-9*(e_4,e_5,e_6)

In [40]:
alpha_cd={('e_3','e_6'):5,('e_4','e_5'):-9}
alpha=g.ext_alg.elt_from_cd({})
alpha+=g.ext_alg.elt_from_cd(alpha_cd)
X_alpha=C.elt({})
for i in range(6):
    temp=alpha
    for j in range(i):
        temp=-X.ad(temp,mod='E')
    X_alpha=X_alpha+temp.tensor(g.basis[9-i])

In [41]:
compute_eigenval(E,alpha)

-2

In [42]:
X_alpha

3*(e_1,e_4,e_2)+6*(e_1,e_5,e_3)+5*(e_1,e_6,e_4)-3*(e_2,e_3,e_2)-3*(e_2,e_4,e_3)+(e_2,e_5,e_4)+5*(e_2,e_6,e_5)-4*(e_3,e_4,e_4)-4*(e_3,e_5,e_5)+5*(e_3,e_6,e_6)-9*(e_4,e_5,e_6)

In [43]:
nW-X_alpha

0

In [44]:
for i in range(6):
    ei=[e1,e2,e3,e4,e5,e6][6-i-1]
    w=3-ei.wght+i
    print('\n',ei)
    for A in g.ext_alg.basis(2,w):
        if set([str(a) for a in A.components]).issubset({'e_1','e_2','e_3','e_4','e_5','e_6'}):
            temp=A
            for j in range(i):
                temp=-X.ad(temp,mod='E')
            print(A,'-->',temp)


 e_6
(e_3,e_6) --> (e_3,e_6)
(e_4,e_5) --> (e_4,e_5)

 e_5
(e_3,e_6) --> (e_2,e_6)+(e_3,e_5)
(e_4,e_5) --> (e_3,e_5)

 e_4
(e_3,e_6) --> (e_1,e_6)+2*(e_2,e_5)+(e_3,e_4)
(e_4,e_5) --> (e_2,e_5)+(e_3,e_4)

 e_3
(e_3,e_6) --> 3*(e_1,e_5)+3*(e_2,e_4)
(e_4,e_5) --> (e_1,e_5)+2*(e_2,e_4)

 e_2
(e_3,e_6) --> 6*(e_1,e_4)+3*(e_2,e_3)
(e_4,e_5) --> 3*(e_1,e_4)+2*(e_2,e_3)

 e_1
(e_3,e_6) --> 9*(e_1,e_3)
(e_4,e_5) --> 5*(e_1,e_3)
